# EVA Phase 2 — Google Colab Training

**UnifiedMultidimensionalTransformerV2** — 20.5M params, 384-dim, 12 layers,
24 heads (6 groups), BPE vocab 4101, AttractorField.

Запускает Phase 2 training на GPU Colab (T4/V100/A100).
По умолчанию B=64, L=128 — использует ~12 GB VRAM из 16.

## Перед запуском

1. Загрузите папку `FCF` на Google Drive в `/MyDrive/EVA/`
   (включая `real_data/`, `eva/`, `train_phase2.py`, `eval_phase2.py`,
   `requirements.txt`, `checkpoints/`)
2. Или укажите ID файлов на Drive в ячейке Data Download
3. Runtime → Change runtime type → **T4 GPU**
4. Run all

In [ ]:
# @title 1. Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# @title 2. Копируем проект с Drive
import os, shutil

SRC = '/content/drive/MyDrive/EVA/FCF'  # откуда копируем
DST = '/content/EVA'                     # куда

if not os.path.exists(SRC):
    # fallback: свежий clone из репозитория
    print(f'{SRC} не найден, клонирую из GitHub...')
    !git clone https://github.com/BlackCatSpb/FCF.git {DST}
    print('Клонировано. Загрузите данные отдельно (ячейка 3).')
else:
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST, ignore=shutil.ignore_patterns('__pycache__', '.git', '.gitignore', '_archive'))
    print(f'Скопировано {SRC} -> {DST}')

%cd {DST}
!ls -la

In [ ]:
# @title 3. Data Download (если данных нет)
import os, glob

DATA_DIR = '/content/EVA/real_data'
need = ['full_corpus_bpe_boundary.npy', 'full_corpus_bpe_labels.npy',
        'full_corpus_bpe.npy', 'bpe_tokenizer.json']
missing = [f for f in need if not os.path.exists(f'{DATA_DIR}/{f}')]

if missing:
    print(f'Отсутствуют: {missing}')
    print('Варианты:')
    print('  1. Загрузите файлы через Colab File Upload (папка real_data)')
    print('  2. Укажите ID файлов на Google Drive ниже')
    print('  3. Конвертируйте из .txt (долгий способ)')

    # Раскомментируйте и укажите ID файлов с Google Drive:
    # FILE_IDS = {
    #     'full_corpus_bpe_boundary.npy': '1ABC...',
    #     'full_corpus_bpe_labels.npy':   '2DEF...',
    #     'full_corpus_bpe.npy':          '3GHI...',
    # }
    # !pip install gdown -q
    # import gdown
    # for fname, fid in FILE_IDS.items():
    #     gdown.download(f'https://drive.google.com/uc?id={fid}',
    #                    f'{DATA_DIR}/{fname}', quiet=False)

    # Или конвертация из .txt:
    # txt = f'{DATA_DIR}/full_corpus_ru.txt'
    # if os.path.exists(txt):
    #     print('Кодирую BPE из .txt...')
    #     %run encode_bpe_corpus.py
    #     print('Готово')
else:
    for f in need:
        size_mb = os.path.getsize(f'{DATA_DIR}/{f}') / 1e6
        print(f'  {f}: {size_mb:.0f} MB OK')
    print('Все данные на месте')

In [ ]:
# @title 4. Устанавливаем зависимости
!pip install -r /content/EVA/requirements.txt -q
!pip install tokenizers -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_mem / 1e9:.1f} GB')

In [ ]:
# @title 5. Проверяем модель и данные
%cd /content/EVA
import sys
sys.path.insert(0, '/content/EVA')
from eva.symbolic.phase1_model import UnifiedMultidimensionalTransformerV2
from eva.symbolic.bpe_tokenizer import BPEVocab

model = UnifiedMultidimensionalTransformerV2(vocab_size=4101)
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')

import numpy as np
ids = np.load('real_data/full_corpus_bpe_boundary.npy')
labels = np.load('real_data/full_corpus_bpe_labels.npy')
print(f'Data: {len(ids):,} tokens, labels range [{labels.min()}, {labels.max()}]')

B, L = 64, 128  # Colab optimised
x = torch.randint(6, 4095, (B, L))
h, scores, weights, heads = model(x, return_scores=True, return_heads=True)
print(f'Forward OK: h={list(h.shape)}, scores={list(scores.shape)}')
print(f'Attractors: {heads["attractor_n_attractors"]}')

In [ ]:
# @title 6. Конфиг Phase 2 — Colab
# Настройки под T4 16GB. Можно менять.

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, os, sys, time

sys.path.insert(0, '/content/EVA')
from eva.symbolic.phase1_model import UnifiedMultidimensionalTransformerV2, D_MODEL
from eva.symbolic.bpe_tokenizer import BPEVocab

# ─── Colab Config ───
N_STEPS = 200000
B = 64           # T4 16GB: B=64, L=128 (~12 GB VRAM)
L = 128          # или B=96, L=96
LR = 5e-4        # Выше из-за большего батча
WARMUP = 4000
LOG_EVERY = 200
SAVE_EVERY = 20000

W_CE = 1.0
W_NXT = 0.05
W_BOUNDARY = 0.1
W_ALIGN = 0.05
W_ATTRACTOR = 0.01
ATTRACTOR_WARMUP = 1000
UPDATE_ATTRACTORS_EVERY = 10

VOCAB = 4101
SPECIAL_IDS = {0, 1, 2, 3, 4096, 4099, 4100}
CKPT_DIR = '/content/drive/MyDrive/EVA/checkpoints/v4'
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device('cuda')
print(f'Device: {device}')
print(f'Config: B={B} L={L} LR={LR} steps={N_STEPS}')
vr = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'VRAM: {vr:.1f} GB | Estimated usage: {B*L*D_MODEL*4*3/1e9:.1f} GB (activations)')

In [ ]:
# @title 7. Загрузка данных
t0 = time.time()
ids = np.load('real_data/full_corpus_bpe_boundary.npy').astype(np.int64)
labels = np.load('real_data/full_corpus_bpe_labels.npy').astype(np.int64)
N = len(ids)
print(f'Data: {N:,} tokens ({time.time()-t0:.1f}s)')
print(f'  vocab range: {ids.min()}..{ids.max()}')

In [ ]:
# @title 8. Инициализация модели
t0 = time.time()
model = UnifiedMultidimensionalTransformerV2(vocab_size=VOCAB).to(device)
total_params = sum(p.numel() for p in model.parameters())
model.train()
print(f'Model: {total_params:,} params ({time.time()-t0:.1f}s)')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
warmup_sched = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-4, total_iters=WARMUP)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_STEPS - WARMUP)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, [warmup_sched, cosine_sched], milestones=[WARMUP])

cv = BPEVocab()

In [ ]:
# @title 9. Training Loop
# ⚠️ Долгий запуск. Runtime → Factory reset runtime, если нужно прервать.

t0 = time.time()
last_save = time.time()

for step in range(N_STEPS):
    idx = np.random.randint(0, N - L - 1, size=B)
    batch_ids = np.stack([ids[i:i+L] for i in idx])
    batch_labels = np.stack([labels[i:i+L] for i in idx])
    x = torch.tensor(batch_ids, dtype=torch.long, device=device)
    y_labels = torch.tensor(batch_labels, dtype=torch.long, device=device)
    targets = torch.tensor(
        np.stack([ids[i+1:i+L+1] for i in idx]),
        dtype=torch.long, device=device)

    optimizer.zero_grad()
    h, scores, weights, heads_out = model.forward(
        x, return_scores=True, return_heads=True, capture_attn=False,
        update_attractors=(step % UPDATE_ATTRACTORS_EVERY == 0))

    # 1. CE loss
    logits = scores
    special_t = torch.tensor(list(SPECIAL_IDS), device=device)
    ce_mask = ~torch.isin(targets, special_t)
    if ce_mask.any():
        ce_loss = F.cross_entropy(
            logits.reshape(-1, logits.shape[-1])[ce_mask.reshape(-1)],
            targets.reshape(-1)[ce_mask.reshape(-1)])
    else:
        ce_loss = torch.tensor(0.0, device=device)

    # 2. nxt loss
    nxt = heads_out.get('boundary_next')
    if nxt is not None and h.shape[1] > 1:
        delta = h[:, 1:] - h[:, :-1]
        nxt_loss = F.mse_loss(nxt[:, :-1], delta)
    else:
        nxt_loss = torch.tensor(0.0, device=device)

    # 3. Boundary loss
    boundary_logits = heads_out.get('boundary_detect')
    if boundary_logits is not None:
        boundary_mask = y_labels >= 0
        boundary_loss_val = F.cross_entropy(
            boundary_logits.reshape(-1, 3)[boundary_mask.reshape(-1)],
            y_labels.reshape(-1)[boundary_mask.reshape(-1)])
    else:
        boundary_loss_val = torch.tensor(0.0, device=device)

    # 4. L_align
    boundary_probs = boundary_logits.softmax(-1) if boundary_logits is not None else None
    if boundary_probs is not None and weights is not None:
        char_inside = boundary_probs[..., 1]
        word_pooled = heads_out.get('boundary_end', h.mean(dim=-1, keepdim=True).expand(-1, -1, D_MODEL))
        word_avg = word_pooled.mean(dim=-1).sigmoid()
        align_loss = F.mse_loss(char_inside.reshape(-1), word_avg.reshape(-1))
    else:
        align_loss = torch.tensor(0.0, device=device)

    # 5. Attractor loss
    af = model.attractor_field
    if step > ATTRACTOR_WARMUP and af.n_attractors > 0:
        valid = af.valid_mask[:af.n_attractors]
        if valid.any():
            centers = af.centers[:af.n_attractors][valid]
            z_flat = h.reshape(-1, D_MODEL)
            dists = torch.cdist(z_flat, centers)
            nearest = dists.argmin(dim=-1)
            attractor_loss = F.mse_loss(z_flat, centers[nearest].detach())
            valid_n = int(valid.sum().item())
            if af.n_attractors > 0 and valid_n > 1:
                c_norm = F.normalize(centers, dim=-1)
                cos_sim = c_norm @ c_norm.T
                mask = 1.0 - torch.eye(valid_n, device=device)
                diversity_loss = (cos_sim * mask).pow(2).mean()
            else:
                diversity_loss = torch.tensor(0.0, device=device)
        else:
            attractor_loss = torch.tensor(0.0, device=device)
            diversity_loss = torch.tensor(0.0, device=device)
    else:
        attractor_loss = torch.tensor(0.0, device=device)
        diversity_loss = torch.tensor(0.0, device=device)

    total = (W_CE * ce_loss + W_NXT * nxt_loss +
             W_BOUNDARY * boundary_loss_val + W_ALIGN * align_loss +
             W_ATTRACTOR * attractor_loss + W_ATTRACTOR * diversity_loss)
    total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if ce_mask.any():
        acc = (logits.argmax(-1)[ce_mask] == targets[ce_mask]).float().mean().item()
    else:
        acc = 0.0

    if boundary_logits is not None:
        bm = y_labels >= 0
        b_acc = (boundary_logits.argmax(-1)[bm] == y_labels[bm]).float().mean().item()
    else:
        b_acc = 0.0

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']
        n_att = heads_out.get('attractor_n_attractors', 0)
        steps_per_sec = (step + 1) / (elapsed + 1e-8)
        eta = (N_STEPS - step) / (steps_per_sec + 1e-8) / 3600
        print(f'[PHASE2 {step}/{N_STEPS}] ce={ce_loss.item():.3f} '
              f'nxt={nxt_loss.item():.3f} bc={boundary_loss_val.item():.3f} '
              f'ac={attractor_loss.item():.3f} dv={diversity_loss.item():.3f} '
              f'acc={acc:.3f} b_acc={b_acc:.3f} att={n_att} '
              f'steps/s={steps_per_sec:.1f} ETA={eta:.1f}h'
              f' | {elapsed/60:.0f}min')

    if step > 0 and step % SAVE_EVERY == 0:
        out_path = f'{CKPT_DIR}/phase2_step_{step}.pt'
        tmp = out_path + '.tmp'
        torch.save({
            'step': step, 'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
        }, tmp)
        os.replace(tmp, out_path)
        print(f'  Saved {out_path} ({time.time()-last_save:.0f}s)')
        last_save = time.time()

elapsed = time.time() - t0
out_path = f'{CKPT_DIR}/phase2_final.pt'
torch.save({
    'step': N_STEPS, 'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
}, out_path)
print(f'Done. {N_STEPS} steps in {elapsed/60:.1f} min. -> {out_path}')

---
## Eval (после обучения)

Запустите ячейки ниже, чтобы проверить качество модели:
- Boundary accuracy на held-out данных
- Token prediction accuracy
- Attractor state (количество, распределение)
- Генерация текста (standard + attractor режимы)

In [ ]:
# @title 10. Eval — Boundary & Token Accuracy
ckpt_path = f'{CKPT_DIR}/phase2_step_{SAVE_EVERY}.pt'  # последний чекпоинт
if not os.path.exists(ckpt_path):
    # fallback: ищем самый свежий
    import glob
    ckpts = sorted(glob.glob(f'{CKPT_DIR}/phase2_step_*.pt'))
    ckpt_path = ckpts[-1] if ckpts else None

if ckpt_path and os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state['model_state'])
    model.eval()
    print(f'Loaded: {ckpt_path} (step {state.get("step","?")})')
else:
    print('No checkpoint found, using current model state')

# Boundary accuracy
rng = np.random.RandomState(42)
n_batches = 20
b_correct, b_total = 0, 0
t_correct, t_total = 0, 0
for _ in range(n_batches):
    idx = rng.randint(0, N - L - 1, size=B)
    batch = np.stack([ids[i:i+L] for i in idx])
    batch_labels = np.stack([labels[i:i+L] for i in idx])
    batch_targets = np.stack([ids[i+1:i+L+1] for i in idx])
    x = torch.tensor(batch, dtype=torch.long, device=device)
    y_lab = torch.tensor(batch_labels, dtype=torch.long, device=device)
    y_tok = torch.tensor(batch_targets, dtype=torch.long, device=device)
    with torch.no_grad():
        h, scores, weights, heads = model(x, return_scores=True, return_heads=True)
    bd = heads.get('boundary_detect')
    if bd is not None:
        valid = y_lab >= 0
        b_correct += (bd.argmax(-1)[valid] == y_lab[valid]).sum().item()
        b_total += valid.sum().item()
    special_t = torch.tensor(list(SPECIAL_IDS), device=device)
    mask = ~torch.isin(y_tok, special_t)
    if mask.any():
        t_correct += (scores.argmax(-1)[mask] == y_tok[mask]).sum().item()
        t_total += mask.sum().item()

print(f'Boundary accuracy: {b_correct/b_total:.4f} ({b_correct}/{b_total})' if b_total > 0 else 'N/A')
print(f'Token accuracy:    {t_correct/t_total:.4f} ({t_correct}/{t_total})' if t_total > 0 else 'N/A')

In [ ]:
# @title 11. Eval — Attractor State
af = model.attractor_field
print(f'Attractors: {af.n_attractors}/{af.max_attractors}')
if af.n_attractors > 1:
    valid = af.valid_mask[:af.n_attractors]
    centers = af.centers[:af.n_attractors][valid]
    counts = af.counts[:af.n_attractors][valid]
    print(f'  Count range: {counts.min().item():.1f}..{counts.max().item():.1f}')
    print(f'  Count mean: {counts.mean().item():.1f}')
    pairwise = torch.cdist(centers, centers, p=2)
    triu = torch.triu(pairwise, diagonal=1)
    if triu[triu>0].numel() > 0:
        print(f'  Mean inter-attractor dist: {triu[triu>0].mean().item():.4f}')
        print(f'  Min inter-attractor dist: {triu[triu>0].min().item():.4f}')
elif af.n_attractors == 1:
    print('  (single attractor — diversity loss inactive)')
else:
    print('  (empty — attractor warmup not yet reached)')

# Dimension utilization
rng = np.random.RandomState(123)
idx = rng.randint(0, N - L - 1, size=B)
batch = np.stack([ids[i:i+L] for i in idx])
x = torch.tensor(batch, dtype=torch.long, device=device)
with torch.no_grad():
    h, _, _, _ = model(x, return_heads=True)
dim_var = h.cpu().numpy().var(axis=(0, 1))
active = (dim_var > dim_var.mean() * 0.5).sum()
print(f'Active dims (var>0.5*mean): {active}/384')

In [ ]:
# @title 12. Eval — Generation
def generate(model, prompt_ids, max_new=64, temp=0.8, use_attractors=False):
    model.eval()
    ids = list(prompt_ids)
    with torch.no_grad():
        for _ in range(max_new):
            inp = torch.tensor([ids[-128:]], dtype=torch.long, device=device)
            h, _, _, heads_out = model(inp, return_heads=True, capture_attn=True)
            z_curr = h[0, -1]
            if use_attractors and model.attractor_field.n_attractors > 0:
                nxt_dir = model.attractor_field.nxt_direction(z_curr.unsqueeze(0))[0]
                z_pred = z_curr + nxt_dir
            else:
                end, nxt, conn = model.boundary_predictor(h[:, -1:])
                z_pred = z_curr + nxt[0, 0]
            logits_know = model.decoder(z_pred.unsqueeze(0).unsqueeze(0))[0, 0]
            sym_coords = model.embed.weight
            dists = -torch.cdist(z_pred.unsqueeze(0), sym_coords, p=2).squeeze(0)
            meta_w = model.meta_weighter(h.mean(dim=1))[0]
            concept = heads_out['concept'][0, -1].item()
            contra = heads_out['contradiction'][0, -1].item()
            logits_conc = dists * (1.0 + concept)
            logits_contr = dists * (1.0 - contra * 0.5)
            final = (meta_w[0]*logits_know + meta_w[1]*logits_conc + meta_w[2]*logits_contr) / temp
            final[:4] = -float('inf')
            for sid in [157, 158, 159, 160, cv.GAP_FILLER_IDX]:
                if sid < len(final):
                    final[sid] = -float('inf')
            for t, c in __import__('collections').Counter(ids).items():
                if t < len(final):
                    final[t] -= c * 0.5
            sv, si = final.sort(descending=True)
            k = min(20, len(sv))
            p = F.softmax(sv[:k], dim=-1)
            nt = si[:k][torch.multinomial(p, 1)].item()
            ids.append(nt)
            if nt in {cv.EOS_IDX, cv.SENT_CLOSE_IDX}:
                break
    return cv.decode(ids), {}

prompts = [cv.encode('Литва'), cv.encode('Россия'), cv.encode('Искусственный')]
print('=== Standard generation ===')
for p in prompts:
    text, _ = generate(model, p, max_new=48, use_attractors=False)
    print(f'  {text[:200]}')
print('\n=== Attractor generation ===')
for p in prompts[:1]:
    text, _ = generate(model, p, max_new=48, use_attractors=True)
    print(f'  {text[:200]}')

In [ ]:
# @title 13. Сохранение результата обратно на Drive
import shutil
backup = '/content/drive/MyDrive/EVA/FCF/checkpoints/v4/'
os.makedirs(backup, exist_ok=True)
for f in os.listdir(CKPT_DIR):
    if f.endswith('.pt'):
        shutil.copy2(f'{CKPT_DIR}/{f}', backup)
        print(f'Copied {f} to Drive')
print('Done')